In [ ]:
from project_utils import load_table, input_path, output_path, calculateCI, mean_error, standard_error, percentile_column, positive_fraction, SEED
import numpy as np
np.random.seed(SEED)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import scipy


In [ ]:
f1 = load_table('citing_papers')
f2 = load_table('nobel_papers')
f3 = load_table('nobel_metrics')


In [ ]:
f1['Nobelyear'] = f1['Nobel'].map(f2.set_index('paperid')['paperyear'])
f1['gap'] = f1['paperyear'] - f1['Nobelyear']
f1['type'] = f1['type'].map({'CC':'CCP','DC':'DCP'})


In [ ]:
f1


In [ ]:
if f2['paperid'].duplicated().any():
    raise ValueError('Nobel paper IDs must be unique')
f3['Field'] = f3['paperid'].map(f2.set_index('paperid')['Field'])
f31 = f3[f3['Field'] == 'Physics']
f32 = f3[f3['Field'] == 'Chemistry']
f33 = f3[f3['Field'] == 'Medicine']


# FIG 2


In [ ]:
scipy.stats.ks_2samp(f31['ni'] , f31['nj'])


In [ ]:
scipy.stats.ks_2samp(f32['ni'] , f32['nj'])


In [ ]:
scipy.stats.ks_2samp(f33['ni'] , f33['nj'])


In [ ]:
sns.set_theme(style="ticks")
plt.subplots(3,1,figsize=(12,22))
for _ in range(3):
    f = [f31 , f32 , f33][_]
    ylabel = ['Physics' , 'Chemistry' , 'Medicine'][_]
    ax = plt.subplot(3,1,_ + 1)
    sns.lineplot(data=f,x="paperyear", y="ni", markers=["o"],label = '#DCP')
    sns.lineplot(data=f,x="paperyear", y="nj", markers=["D"],label = '#CCP')

    ax.yaxis.grid(True)
    ax.set_yscale("log")
    plt.ylim(10,50000)
    plt.xticks(size = 28)
    plt.yticks([10,100,1000,10000],size = 28)
    plt.legend(fontsize = 28)
    plt.ylabel(ylabel,fontsize = 32)
    plt.xlabel(r'',fontsize = 0)
plt.xlabel(r'publishing year',fontsize = 32)


# fig 2


In [ ]:
x_pos = [1,2,3,5,6,7,]
d1 = f31['ni']
d2 = f32['ni']
d3 = f33['ni']
d5 = f31['nj']
d6 = f32['nj']
d7 = f33['nj']
d = [np.mean(d1),np.mean(d2),np.mean(d3),np.mean(d5),np.mean(d6),np.mean(d7)]
std = [standard_error(d1),standard_error(d2),standard_error(d3),standard_error(d5),standard_error(d6),standard_error(d7)]
colors = ["#154360","#1F618D","#2980B9","#784212","#AF601A","#E67E22"]


In [ ]:
f, ax = plt.subplots(figsize=(14, 9))
plt.bar(x_pos, d, yerr=std,align='center', color= colors,alpha=0.9,capsize=10)
ax.yaxis.grid(True)
ax.set_yscale("log")
plt.xlim(0,8)
plt.xticks(x_pos,['Physics','Chemistry','Medicine','Physics','Chemistry','Medicine'],fontsize = 23)
for _ in range(len(d)):
    plt.text(x_pos[_],d[_]*0.55 , str(np.round(d[_] , 1)),fontsize = 25,c = 'white', ha='center', fontweight='bold')
plt.yticks([10,100,1000,10000,100000],size = 28)
plt.ylabel(r'Number',fontsize = 35)
ax.axvspan(min(x_pos)-1, max(x_pos) / 2+0.5, facecolor='#154360', alpha=0.2)
ax.axvspan(max(x_pos) / 2+0.5, max(x_pos)+1, facecolor='#6E2C00', alpha=0.2)


In [ ]:
print(scipy.stats.mannwhitneyu(d1 , d5, alternative='two-sided'))
print(scipy.stats.mannwhitneyu(d2 , d6, alternative='two-sided'))
print(scipy.stats.mannwhitneyu(d3 , d7, alternative='two-sided'))


In [ ]:
x_pos = [1,2,3,]
d1 = f2.loc[f2['Field'].eq('Physics'), 'cit']
d2 = f2.loc[f2['Field'].eq('Chemistry'), 'cit']
d3 = f2.loc[f2['Field'].eq('Medicine'), 'cit']
d = [np.mean(d1),np.mean(d2),np.mean(d3)]
std = [standard_error(d1),standard_error(d2),standard_error(d3)]
colors = ["#641E16","#922B21","#C0392B"]


In [ ]:
f, ax = plt.subplots(figsize=(8, 12))
plt.bar(x_pos, d, yerr=std,align='center', color= colors,alpha=0.9,capsize=10)
ax.yaxis.grid(True)
ax.set_yscale("log")
plt.xticks(x_pos,['Physics','Chemistry','Medicine'],fontsize = 30)
for _ in range(len(d)):
    plt.text(x_pos[_],d[_]*0.55 , str(np.round(d[_] , 1)),fontsize = 25,c = 'white', ha='center', fontweight='bold')
plt.yticks([10,100,1000,10000,100000],size = 28)
plt.ylabel(r'Avg. Citation of Nobel',fontsize = 35)


In [ ]:
sns.set_theme(style="ticks")
f, ax = plt.subplots(figsize=(10, 6))
g = sns.lineplot(
    data=f2,x="paperyear",c='#641E16', y="cit")
ax.yaxis.grid(True)
ax.set_yscale("log")
plt.xticks(size = 28)
plt.yticks(size = 28)
plt.ylabel(r'Avg. Citation of Nobel',fontsize = 35)
plt.xlabel(r'publishing year',fontsize = 35)


In [ ]:
d = f2['paperyear'].groupby(f2['paperyear']).count()

f, ax = plt.subplots(figsize=(10, 6))
ax.plot(d.index, d.values, linewidth=2.0, c = "#212F3C")
ax.yaxis.grid(True)
plt.xticks(size = 28)
plt.yticks(size = 28)
plt.ylabel(r'Number of Nobel',fontsize = 35)
plt.xlabel(r'publishing year',fontsize = 35)
